# Predykcja bankructwa polskich firm — pipeline bez wycieku danych

**Zbiór:** `5year.arff`, czyli prognoza bankructwa w następnym roku.  
**Cel:** klasyfikacja `class=1` (bankructwo).  
**Metodologia:** rozdzielenie treningu i testu przed dopasowaniem imputacji, winsoryzacji i skalowania; stratyfikowana walidacja krzyżowa na treningu; metryka główna Average Precision (PR-AUC).  
**Uwaga:** plik Excel `Dane_ML_gotowe` nie jest używany do testu, gdyż zawiera mediany i granice winsoryzacji wyznaczone na całych danych.

Kod został przygotowany dla scikit-learn ≥ 1.2 i danych lokalnych, bez połączenia z internetem.

## 1. Importy i wczytanie surowego zbioru

Nie mieszamy `1year`–`5year`: są to różne horyzonty prognozy, a bez identyfikatorów firm nie możemy wykluczyć zależności między nimi.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import arff

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    precision_recall_curve, precision_score, recall_score,
    fbeta_score, balanced_accuracy_score, confusion_matrix,
    classification_report
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate,
    RandomizedSearchCV, cross_val_predict
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from xgboost import XGBClassifier


RANDOM_STATE = 42
PLIK = Path("5year.arff")  

rekordy, meta = arff.loadarff(str(PLIK))
kolumny = [n for n in rekordy.dtype.names if n != "class"]
X = pd.DataFrame({n: rekordy[n].astype(float) for n in kolumny})
y = pd.Series([int(k.decode("ascii")) for k in rekordy["class"]], name="class")

# Identyczne wiersze (łącznie z etykietą) usuwamy przed podziałem.
dane = X.copy()
dane["class"] = y
print("Surowych rekordów:", len(dane))
print("Duplikaty X + y:", int(dane.duplicated().sum()))
dane = dane.drop_duplicates().reset_index(drop=True)
y = dane.pop("class").astype(int)
X = dane
print("Po usunięciu duplikatów:", X.shape)
print("Bankructwa:", int(y.sum()), "udział:", round(y.mean() * 100, 2), "%")

Surowych rekordów: 5910
Duplikaty X + y: 60
Po usunięciu duplikatów: (5850, 64)
Bankructwa: 408 udział: 6.97 %


## 2. Wstępna diagnostyka

Diagnoza służy zrozumieniu problemu. Przy wyborze hiperparametrów, cech, progów lub strategii imputacji używamy tylko danych treningowych.

In [2]:
raport_brakow = (
    X.isna().mean().mul(100).sort_values(ascending=False)
    .rename("odsetek_brakow_pct").to_frame()
)
print("Odsetek brakujących komórek:", round(X.isna().to_numpy().mean() * 100, 2), "%")
print("Przykładowe kolumny z największą liczbą braków:")
print(raport_brakow.head(10).to_string())
print("Dokładnie zduplikowane wiersze po czyszczeniu:", int(X.duplicated().sum()))
print("Liczba cech:", X.shape[1])

Odsetek brakujących komórek: 1.24 %
Przykładowe kolumny z największą liczbą braków:
        odsetek_brakow_pct
Attr37           43.162393
Attr27            6.666667
Attr45            4.547009
Attr60            4.547009
Attr24            2.307692
Attr28            1.811966
Attr64            1.811966
Attr53            1.811966
Attr54            1.811966
Attr21            1.760684
Dokładnie zduplikowane wiersze po czyszczeniu: 0
Liczba cech: 64


## 3. Najpierw test, potem każda operacja ucząca się na danych

Zbiór testowy (20%) pozostaje nietknięty do zakończenia doboru modeli, hiperparametrów i progu. `StratifiedKFold` zachowuje częstość bankructw w częściach treningowych.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "pozytywne:", int(y_train.sum()))
print("Test:", X_test.shape, "pozytywne:", int(y_test.sum()))

Train: (4680, 64) pozytywne: 326
Test: (1170, 64) pozytywne: 82


## 4. Przetwarzanie wewnątrz Pipeline

- `QuantileClipper` zapamiętuje percentyle tylko ze zbioru podanego do `fit` (w CV: wyłącznie fold treningowy).
- `SimpleImputer(strategy="median", add_indicator=True)` uzupełnia braki i dodaje automatycznie flagi brakujących danych.
- `RobustScaler` jest stosowany **tylko do regresji logistycznej**. Drzewa nie wymagają skalowania.
- W obecnym ARFF nie ma kategorycznych predyktorów. Gdyby pojawiły się w innych danych, obsłuży je `OneHotEncoder`.

W obu gałęziach `ColumnTransformer` dane są przetwarzane wyłącznie na podstawie treningu.

In [4]:
class QuantileClipper(BaseEstimator, TransformerMixin):
    """Winsoryzacja każdej kolumny do percentyli obliczonych w fit()."""
    def __init__(self, lower=0.01, upper=0.99):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        a = np.asarray(X, dtype=float)
        self.lower_ = np.nanquantile(a, self.lower, axis=0)
        self.upper_ = np.nanquantile(a, self.upper, axis=0)
        return self

    def transform(self, X):
        a = np.asarray(X, dtype=float)
        return np.clip(a, self.lower_, self.upper_)

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

def przygotowanie(skalowanie=False, winsoryzacja=True):
    kroki_num = []
    if winsoryzacja:
        kroki_num.append(("clip", QuantileClipper(0.01, 0.99)))
    kroki_num.append(("impute", SimpleImputer(strategy="median", add_indicator=True)))
    if skalowanie:
        kroki_num.append(("scale", RobustScaler()))

    transformatory = [("numeric", Pipeline(kroki_num), num_cols)]
    if cat_cols:
        transformatory.append(("categoric", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols))
    return ColumnTransformer(
        transformatory, remainder="drop", sparse_threshold=0.0
    )

def potok(model, skalowanie=False, winsoryzacja=True):
    return Pipeline([
        ("preprocess", przygotowanie(skalowanie, winsoryzacja)),
        ("model", model)
    ])

## 5. Baseline i 4 modele — porównanie 5-fold CV

Najważniejsza metryka porównawcza: `average_precision` (AP), zbliżona interpretacyjnie do pola pod krzywą Precision–Recall. Wartość AP dla klasyfikatora losowego/stałego jest zbliżona do częstości klasy pozytywnej. `ROC-AUC` jest miarą dodatkową.

Wyniki pochodzą wyłącznie z `X_train`; **nie dotykamy `X_test`**.

In [5]:

modele = {
    "Dummy (baseline)": potok(DummyClassifier(strategy="prior")),

    "HistGradientBoosting": potok(HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
        min_samples_leaf=30, l2_regularization=1.0,
        random_state=RANDOM_STATE
    )),

    "LogisticRegression": potok(LogisticRegression(
        C=0.2, class_weight="balanced", max_iter=3000,
        solver="lbfgs", random_state=RANDOM_STATE
    ), skalowanie=True),

    "RandomForest": potok(RandomForestClassifier(
        n_estimators=250, max_features="sqrt", min_samples_leaf=3,
        class_weight="balanced_subsample", n_jobs=-1,
        random_state=RANDOM_STATE
    )),

    "XGBoost": potok(XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        scale_pos_weight=1,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=2
    ))
}

# Walidacja krzyżowa dla wszystkich modeli
wyniki = []

for nazwa, estimator in modele.items():
    s = cross_validate(
        estimator, X_train, y_train,
        cv=cv,
        scoring={
            "ap": "average_precision",
            "roc": "roc_auc"
        },
        n_jobs=1,
        error_score="raise"
    )

    wyniki.append({
        "model": nazwa,
        "AP_mean": s["test_ap"].mean(),
        "AP_std": s["test_ap"].std(ddof=1),
        "ROC_AUC_mean": s["test_roc"].mean()
    })

wyniki_cv = pd.DataFrame(wyniki).sort_values(
    "AP_mean", ascending=False
)

print(wyniki_cv.to_string(index=False))
print("Przewaga nad baseline musi być oceniana w CV; nie na accuracy.")


               model  AP_mean   AP_std  ROC_AUC_mean
HistGradientBoosting 0.784081 0.038778      0.951521
             XGBoost 0.764439 0.049376      0.947008
  LogisticRegression 0.561264 0.075240      0.903502
        RandomForest 0.543421 0.058761      0.922973
    Dummy (baseline) 0.069658 0.000478      0.500000
Przewaga nad baseline musi być oceniana w CV; nie na accuracy.


## 6. Strojenie kandydata nr 1 (HGB)

To **przykładowa niewielka przestrzeń hiperparametrów**. W większym badaniu należy zwiększyć liczbę losowań i rozważyć walidację zewnętrzną/nested CV. `class_weight` jest strojonym wariantem, a nie obowiązkowym włączeniem.

In [6]:
hgb = modele["HistGradientBoosting"]
parametry = {
    "model__max_iter": [120, 200, 280],
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_leaf_nodes": [7, 15, 31],
    "model__min_samples_leaf": [20, 40, 60],
    "model__l2_regularization": [0.0, 1.0, 10.0],
    "model__class_weight": [None, "balanced"]
}
search = RandomizedSearchCV(
    hgb, param_distributions=parametry,
    n_iter=25, scoring="average_precision",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    refit=True, random_state=RANDOM_STATE, n_jobs=1,
    error_score="raise", return_train_score=False
)
search.fit(X_train, y_train)
print("Najlepsze AP w CV:", round(search.best_score_, 4))
print("Wybrane parametry:", search.best_params_)
najlepszy = search.best_estimator_

Najlepsze AP w CV: 0.7763
Wybrane parametry: {'model__min_samples_leaf': 20, 'model__max_leaf_nodes': 15, 'model__max_iter': 120, 'model__learning_rate': 0.05, 'model__l2_regularization': 0.0, 'model__class_weight': None}


In [7]:

# Strojenie hiperparametrów XGBoost

parametry_xgb = {
    "model__n_estimators": [150, 250, 400],
    "model__learning_rate": [0.02, 0.05, 0.1],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__subsample": [0.7, 0.85, 1.0],
    "model__colsample_bytree": [0.7, 0.85, 1.0],
    "model__reg_lambda": [1, 5, 10],
    "model__scale_pos_weight": [1, 3, 7, 13]
}

cv_inner = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

search_xgb = RandomizedSearchCV(
    estimator=modele["XGBoost"],
    param_distributions=parametry_xgb,
    n_iter=25,
    scoring="average_precision",
    cv=cv_inner,
    refit=True,
    random_state=RANDOM_STATE,
    n_jobs=1,
    error_score="raise"
)

search_xgb.fit(X_train, y_train)

print("Najlepszy wynik AP:", search_xgb.best_score_)
print("Najlepsze parametry:")
print(search_xgb.best_params_)


Najlepszy wynik AP: 0.762379741379411
Najlepsze parametry:
{'model__subsample': 0.7, 'model__scale_pos_weight': 7, 'model__reg_lambda': 1, 'model__n_estimators': 150, 'model__min_child_weight': 3, 'model__max_depth': 5, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.85}


In [8]:

cv_outer = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=123
)

for nazwa, procedura in [
    ("HistGradientBoosting", search),
    ("XGBoost", search_xgb)
]:
    ocena = cross_validate(
        procedura,
        X_train,
        y_train,
        cv=cv_outer,
        scoring="average_precision",
        n_jobs=1
    )

    print(
        nazwa,
        "AP:", round(ocena["test_score"].mean(), 4),
        "STD:", round(ocena["test_score"].std(ddof=1), 4)
    )


HistGradientBoosting AP: 0.7896 STD: 0.0208
XGBoost AP: 0.7796 STD: 0.0238


## 7. Wybór progu wyłącznie na treningu

Próg 0,5 często nie odpowiada celowi wykrywania bankructw. Dla demonstracji wybieramy próg maksymalizujący **F2** (większy nacisk na czułość niż na precyzję) na predykcjach out-of-fold. Jeśli znasz koszty fałszywego alarmu i niewykrytego bankructwa, wybierz próg według tych kosztów zamiast F2.

In [9]:
p_oof = cross_val_predict(
    najlepszy, X_train, y_train, cv=cv,
    method="predict_proba", n_jobs=1
)[:, 1]
precyzja, czulosc, progi = precision_recall_curve(y_train, p_oof)
wynik_f2 = (5 * precyzja[:-1] * czulosc[:-1]) / (
    4 * precyzja[:-1] + czulosc[:-1] + 1e-12
)
idx = int(np.argmax(wynik_f2))
prog = float(progi[idx])
print("Próg (F2, out-of-fold):", round(prog, 5))
print("F2 na OOF:", round(float(wynik_f2[idx]), 4))
# Uwaga: do rygorystycznej estymacji skuteczności doboru progu użyj nested CV.
# Tu niezależny test pozostaje nietknięty i zostanie użyty tylko raz.

Próg (F2, out-of-fold): 0.08113
F2 na OOF: 0.7229


## 8. Jednorazowy test końcowy

Nie zmieniaj progu ani hiperparametrów na podstawie wyników na `X_test`: spowodowałoby to dalszą optymalizację pod test.

In [10]:
# RandomizedSearchCV(refit=True) już dopasował model na całym X_train.
p_test = najlepszy.predict_proba(X_test)[:, 1]
pred_test = (p_test >= prog).astype(int)

metryki_test = {
    "AP": average_precision_score(y_test, p_test),
    "ROC_AUC": roc_auc_score(y_test, p_test),
    "Precision": precision_score(y_test, pred_test, zero_division=0),
    "Recall": recall_score(y_test, pred_test, zero_division=0),
    "F2": fbeta_score(y_test, pred_test, beta=2, zero_division=0),
    "Balanced_accuracy": balanced_accuracy_score(y_test, pred_test),
    "Brier_score": brier_score_loss(y_test, p_test)
}
print(pd.Series(metryki_test).round(4))
print("Macierz pomyłek [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, pred_test))
print(classification_report(y_test, pred_test, digits=3, zero_division=0))

AP                   0.8474
ROC_AUC              0.9678
Precision            0.5224
Recall               0.8537
F2                   0.7576
Balanced_accuracy    0.8974
Brier_score          0.0225
dtype: float64
Macierz pomyłek [[TN, FP], [FN, TP]]:
[[1024   64]
 [  12   70]]
              precision    recall  f1-score   support

           0      0.988     0.941     0.964      1088
           1      0.522     0.854     0.648        82

    accuracy                          0.935      1170
   macro avg      0.755     0.897     0.806      1170
weighted avg      0.956     0.935     0.942      1170



## 9. Analizy wrażliwości (wykonać jako oddzielne eksperymenty na treningu)

1. HGB bez winsoryzacji i / lub bez flag braków; ewentualnie HGB przyjmujący NaN bez imputacji.
2. Porównanie `class_weight=None` vs `"balanced"`.
3. Usunięcie `Attr37` (~43% braków), wyłącznie jako ablation study; nie usuwaj automatycznie.
4. PCA / Elastic Net jako **alternatywy** dla modelu liniowego, dopasowane wewnątrz CV.
5. Kalibracja prawdopodobieństw na wydzielonej części treningowej (np. `CalibratedClassifierCV`) i diagnostyka wykresu kalibracyjnego.
6. Walidacja firmowa (`GroupKFold`) i/lub chronologiczna, jeśli uzyskasz identyfikatory przedsiębiorstw i daty. Bez tych metadanych możliwy pozostaje ukryty wyciek i optymistyczna ocena.

**Nie używaj SMOTE przed podziałem**, a w pierwszej kolejności sprawdź wagi klas oraz dobór progu.

**Interpretacja:** po ustaleniu modelu można obliczyć `permutation_importance` na niezależnym zestawie interpretacyjnym albo SHAP (poza podstawowym scikit-learn) i zweryfikować sens ekonomiczny najważniejszych wskaźników.

## 10. Utrwalenie końcowego modelu

Zapisuje dopasowaną całość: winsoryzację, imputację, skalowanie/kodowanie (jeśli dotyczy) oraz estymator. W zastosowaniach produkcyjnych dołącz wersję bibliotek i opis pochodzenia danych.

In [16]:
import joblib
joblib.dump({"pipeline": najlepszy, "threshold": prog,
             "feature_names": list(X.columns)},
            "model_bankructwo_5year.joblib")
print("Zapisano model_bankructwo_5year.joblib")

Zapisano model_bankructwo_5year.joblib


### Test

In [17]:
import joblib
import pandas as pd

# Wczytanie zapisanego modelu
zapis = joblib.load("model_bankructwo_5year.joblib")

model = zapis["pipeline"]
prog = zapis["threshold"]
cechy = zapis["feature_names"]

# Przykładowa predykcja na jednym przedsiębiorstwie ze zbioru testowego
firma = X_test.iloc[[0]][cechy]

prawdopodobienstwo = model.predict_proba(firma)[0, 1]
predykcja = int(prawdopodobienstwo >= prog)

print("Liczba wymaganych wskaźników:", len(cechy))
print("Próg decyzyjny:", round(prog, 5))
print("Wynik modelu:", round(prawdopodobienstwo, 4))
print("Predykcja:", predykcja)

Liczba wymaganych wskaźników: 64
Próg decyzyjny: 0.08113
Wynik modelu: 0.0017
Predykcja: 0
